In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURAÇÕES DA SIMULAÇÃO
# ============================================================

dt = 1e-3                              # Passo de tempo (s)
T_FINAL = 12.0                         # Duração da simulação (s)
t = np.arange(0, T_FINAL, dt)          # Eixo de tempo

# Polos reais do enunciado
P1 = -1.0
P2 = -5.0
P3 = +1.0

# Par complexo de referência: p = -sigma ± j*omega
SIGMA_REF = 1.0
OMEGA_REF = 5.0

# Estudo da parte real (sigma) com omega fixo
OMEGA_FIXO = 5.0
lista_sigma = [3.0, 1.0, 0.3, 0.0, -0.3]

# Estudo da parte imaginária (omega) com sigma fixo
SIGMA_FIXO = 1.0
lista_omega = [2.0, 5.0, 10.0, 20.0]

# Estudo dos zeros (sistema com polos em -1 e -5)
lista_zeros = [None, -10.0, -2.0, -0.5, +2.0]    # None = sem zero


# ============================================================
# 2. MODELAGEM: H(s) = N(s) / D(s) A PARTIR DE POLOS E ZEROS
# ============================================================

def par_complexo(sigma, omega):
    """
    Par complexo conjugado: p = -sigma ± j*omega
    """

    return [complex(-sigma, omega), complex(-sigma, -omega)]


def construir_sistema(polos, zeros=(), K=None):
    """
    Monta H(s) = K * prod(s - z_k) / prod(s - p_k).

    Se K não for informado, ele é escolhido para que H(0) = 1
    (ganho DC unitário), o que facilita comparar as respostas.

    Retorna os coeficientes (num, den) em potências decrescentes de s.
    """

    polos = np.array(polos, dtype=complex)
    zeros = np.array(zeros, dtype=complex)

    if K is None:
        K = np.real(np.prod(-polos) / np.prod(-zeros))

    num = np.atleast_1d(np.real(K * np.poly(zeros)))
    den = np.real(np.poly(polos))

    return num, den


def residuos(num, den):
    """
    Expansão em frações parciais (polos simples):

    H(s) = sum( r_k / (s - p_k) ),   r_k = N(p_k) / D'(p_k)
    """

    polos = np.roots(den)
    r = np.polyval(num, polos) / np.polyval(np.polyder(den), polos)

    return polos, r


def resposta_impulso(t, num, den):
    """
    Resposta ao impulso (transformada inversa de Laplace):

    h(t) = sum( r_k * exp(p_k * t) )
    """

    polos, r = residuos(num, den)

    h = np.zeros(len(t), dtype=complex)

    for pk, rk in zip(polos, r):
        h += rk * np.exp(pk * t)

    return np.real(h)


def resposta_degrau(t, num, den):
    """
    Resposta ao degrau unitário (Y(s) = H(s)/s):

    y(t) = H(0) + sum( (r_k / p_k) * exp(p_k * t) )
    """

    polos, r = residuos(num, den)

    y = np.full(len(t), np.polyval(num, 0) / np.polyval(den, 0), dtype=complex)

    for pk, rk in zip(polos, r):
        y += (rk / pk) * np.exp(pk * t)

    return np.real(y)


def degrau(polos, zeros=(), K=None):
    """
    Atalho: resposta ao degrau a partir dos polos e zeros.
    """

    num, den = construir_sistema(polos, zeros, K)

    return resposta_degrau(t, num, den)


# ============================================================
# 3. SIMULAÇÃO NUMÉRICA (RUNGE-KUTTA DE 4ª ORDEM)
# ============================================================

def simular_rk4(num, den, t):
    """
    Integra o sistema na forma canônica controlável com entrada
    degrau unitário, pelo método de Runge-Kutta de 4ª ordem:

    x' = A x + B u,   y = C x
    """

    # Normaliza para que o denominador seja mônico
    num = num / den[0]
    den = den / den[0]

    n = len(den) - 1

    A = np.zeros((n, n))
    A[:-1, 1:] = np.eye(n - 1)
    A[-1, :] = -den[1:][::-1]

    B = np.zeros(n)
    B[-1] = 1.0

    C = np.zeros(n)
    num_crescente = num[::-1]
    C[:len(num_crescente)] = num_crescente

    h = t[1] - t[0]
    x = np.zeros(n)
    y = np.zeros(len(t))

    def f(x):
        return A @ x + B

    for i in range(len(t)):
        y[i] = C @ x

        k1 = f(x)
        k2 = f(x + 0.5 * h * k1)
        k3 = f(x + 0.5 * h * k2)
        k4 = f(x + h * k3)

        x = x + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)

    return y


# ============================================================
# 4. ESTABILIDADE E MÉTRICAS
# ============================================================

def classificar_estabilidade(polos, tol=1e-9):
    """
    Critério pela posição dos polos (polos simples):

    Re{p_k} < 0 para todo k  -> assintoticamente estável
    algum Re{p_k} = 0        -> marginalmente estável
    algum Re{p_k} > 0        -> instável
    """

    parte_real = np.real(np.array(polos, dtype=complex))

    if np.any(parte_real > tol):
        return "INSTÁVEL"

    if np.any(np.abs(parte_real) <= tol):
        return "MARGINAL"

    return "ESTÁVEL"


def tempo_acomodacao(t, y, valor_final=1.0, tol=0.02):
    """
    Tempo de acomodação (critério de 2%).
    """

    fora = np.where(np.abs(y - valor_final) > tol * abs(valor_final))[0]

    if len(fora) == 0:
        return 0.0

    if fora[-1] + 1 >= len(t):
        return np.nan

    return t[fora[-1] + 1]


def texto_polos(polos):
    """
    Texto com os polos (cada par conjugado aparece uma vez).
    """

    partes = []

    for p in np.array(polos, dtype=complex):
        if abs(p.imag) < 1e-9:
            partes.append(f"{p.real + 0.0:+.2f}")
        elif p.imag > 0:
            partes.append(f"{p.real + 0.0:+.2f}±{p.imag:.2f}j")

    return ", ".join(partes)


# ============================================================
# 5. REPRESENTAÇÃO DOS POLOS NO PLANO-s
# ============================================================

fig, eixo = plt.subplots(figsize=(9, 6))

eixo.axvspan(-8, 0, color="tab:green", alpha=0.08)
eixo.axvspan(0, 3, color="tab:red", alpha=0.08)

eixo.text(-4, 5.7, "Semiplano esquerdo\n(estável)", ha="center", va="top")
eixo.text(1.5, 5.7, "Semiplano direito\n(instável)", ha="center", va="top")

pontos = [
    (P1, 0, "p1 = -1", "C0"),
    (P2, 0, "p2 = -5", "C1"),
    (P3, 0, "p3 = +1", "C3"),
    (-SIGMA_REF, OMEGA_REF, r"$p_{4,5}$ = -1 ± j5", "C2"),
    (-SIGMA_REF, -OMEGA_REF, None, "C2"),
]

for re, im, rotulo, cor in pontos:
    eixo.plot(re, im, "x", color=cor, markersize=12, markeredgewidth=2.5)

    if rotulo is not None:
        eixo.annotate(rotulo, (re, im), textcoords="offset points",
                      xytext=(8, 10), color=cor)

eixo.axhline(0, color="k", linewidth=0.8)
eixo.axvline(0, color="k", linewidth=1.2)

eixo.set_xlim(-8, 3)
eixo.set_ylim(-6, 6)

eixo.set_title("Polos no plano-s")
eixo.set_xlabel(r"Re(s) = $\sigma$")
eixo.set_ylabel(r"Im(s) = j$\omega$")

eixo.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 6. POLOS REAIS ESTÁVEIS: VELOCIDADE DE RESPOSTA
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(t, np.ones_like(t), "k--", linewidth=1, label="Entrada (degrau)")

for polo, cor in [(P1, "C0"), (P2, "C1")]:
    y = degrau([polo])
    plt.plot(t, y, color=cor, label=f"p = {polo:g}  ($\\tau$ = {1/abs(polo):g} s)")

    # Em t = 1/|p| a saída atinge 63,2% do valor final
    plt.plot(1 / abs(polo), 0.632, "o", color=cor, markersize=5)

plt.xlim(0, 8)

plt.title("Polos reais no semiplano esquerdo: quanto mais à esquerda, mais rápida a resposta")
plt.xlabel("t (s)")
plt.ylabel("y(t)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 7. POLO REAL INSTÁVEL (p3 = +1)
# ============================================================

num3, den3 = construir_sistema([P3], K=1.0)       # H(s) = 1 / (s - 1)

fig, eixos = plt.subplots(1, 2, figsize=(12, 5))

t_curto = t[t <= 5]

eixos[0].plot(t_curto, resposta_degrau(t_curto, num3, den3), label=r"$y(t) = e^{t} - 1$")
eixos[0].set_title("Resposta ao degrau")

eixos[1].plot(t_curto, resposta_impulso(t_curto, num3, den3), label=r"$h(t) = e^{t}$")
eixos[1].set_title("Resposta ao impulso")

for eixo in eixos:
    eixo.set_xlabel("t (s)")
    eixo.set_ylabel("Amplitude")
    eixo.legend()
    eixo.grid(True)

plt.suptitle("Polo em +1: componente exponencial crescente (sistema instável)")
plt.tight_layout()
plt.show()


# ============================================================
# 8. PARES COMPLEXOS: EFEITO DA PARTE REAL (SIGMA)
# ============================================================

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

for i, sigma in enumerate(lista_sigma):
    y = degrau(par_complexo(sigma, OMEGA_FIXO))

    if sigma > 0:
        eixo = eixos[0]

        # Envoltória da oscilação: 1 + A * exp(-sigma*t)
        amplitude = np.sqrt(1 + (sigma / OMEGA_FIXO) ** 2)
        eixo.plot(t, 1 + amplitude * np.exp(-sigma * t), "--",
                  color=f"C{i}", alpha=0.5, linewidth=1)
    else:
        eixo = eixos[1]

    eixo.plot(t, y, color=f"C{i}", label=rf"$\sigma$ = {sigma:g}")

eixos[0].set_ylim(0, 2.2)
eixos[0].set_title(r"(a) Semiplano esquerdo ($\sigma$ > 0), com envoltória tracejada")

eixos[1].set_title(r"(b) Eixo j$\omega$ e semiplano direito ($\sigma \leq$ 0)")

for eixo in eixos:
    eixo.set_xlabel("t (s)")
    eixo.set_ylabel("y(t)")
    eixo.legend()
    eixo.grid(True)

plt.suptitle(r"Pares complexos $p = -\sigma \pm j\omega$ com $\omega$ = 5 rad/s")
plt.tight_layout()
plt.show()


# ============================================================
# 9. PARES COMPLEXOS: EFEITO DA PARTE IMAGINÁRIA (OMEGA)
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(t, np.ones_like(t), "k--", linewidth=1, label="Entrada (degrau)")

for i, omega in enumerate(lista_omega):
    y = degrau(par_complexo(SIGMA_FIXO, omega))
    periodo = 2 * np.pi / omega

    plt.plot(t, y, color=f"C{i}",
             label=rf"$\omega$ = {omega:g} rad/s  (T = {periodo:.2f} s)")

plt.xlim(0, 6)

plt.title(r"Pares complexos com $\sigma$ = 1: a parte imaginária define a frequência de oscilação")
plt.xlabel("t (s)")
plt.ylabel("y(t)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 10. MAPA DE POLOS DOS ESTUDOS COMPLEXOS
# ============================================================

fig, eixos = plt.subplots(1, 2, figsize=(12, 5))

for i, sigma in enumerate(lista_sigma):
    p = par_complexo(sigma, OMEGA_FIXO)
    eixos[0].plot(np.real(p), np.imag(p), "x", color=f"C{i}",
                  markersize=10, markeredgewidth=2, label=rf"$\sigma$ = {sigma:g}")

eixos[0].set_title(r"Variando $\sigma$ ($\omega$ = 5 rad/s)")

for i, omega in enumerate(lista_omega):
    p = par_complexo(SIGMA_FIXO, omega)
    eixos[1].plot(np.real(p), np.imag(p), "x", color=f"C{i}",
                  markersize=10, markeredgewidth=2, label=rf"$\omega$ = {omega:g}")

eixos[1].set_title(r"Variando $\omega$ ($\sigma$ = 1)")

for eixo in eixos:
    eixo.axhline(0, color="k", linewidth=0.8)
    eixo.axvline(0, color="k", linewidth=1.2)
    eixo.set_xlabel("Re(s)")
    eixo.set_ylabel("Im(s)")
    eixo.legend()
    eixo.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 11. SISTEMA COMPOSTO: UM POLO INSTÁVEL DOMINA A RESPOSTA
# ============================================================

polos_estavel = [P1, P2] + par_complexo(SIGMA_REF, OMEGA_REF)
polos_instavel = [P1, P2, P3] + par_complexo(SIGMA_REF, OMEGA_REF)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

eixos[0].plot(t, degrau(polos_estavel))
eixos[0].set_title("Polos: -1, -5 e -1 ± j5 (todos com Re < 0)")

t_instavel = t[t <= 6]
num_i, den_i = construir_sistema(polos_instavel)
eixos[1].plot(t_instavel, resposta_degrau(t_instavel, num_i, den_i), color="C3")
eixos[1].set_title("Mesmos polos + p3 = +1 (um polo com Re > 0)")

for eixo in eixos:
    eixo.set_xlabel("t (s)")
    eixo.set_ylabel("y(t)")
    eixo.grid(True)

plt.suptitle("Resposta ao degrau do sistema com todos os polos")
plt.tight_layout()
plt.show()


# ============================================================
# 12. EFEITO DOS ZEROS (POLOS EM -1 E -5)
# ============================================================

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

eixos[0].plot([P1, P2], [0, 0], "kx", markersize=11, markeredgewidth=2.5, label="Polos")

eixos[1].plot(t, np.ones_like(t), "k--", linewidth=1, label="Entrada (degrau)")

for i, zero in enumerate(lista_zeros):
    if zero is None:
        y = degrau([P1, P2])
        rotulo = "sem zero"
    else:
        y = degrau([P1, P2], zeros=[zero])
        rotulo = f"zero em {zero:+g}"
        eixos[0].plot(zero, 0, "o", color=f"C{i}", markersize=9,
                      markerfacecolor="none", markeredgewidth=2, label=rotulo)

    eixos[1].plot(t, y, color=f"C{i}", label=rotulo)

eixos[0].axhline(0, color="k", linewidth=0.8)
eixos[0].axvline(0, color="k", linewidth=1.2)
eixos[0].set_xlim(-12, 4)
eixos[0].set_ylim(-3, 3)
eixos[0].set_title("Mapa de polos e zeros")
eixos[0].set_xlabel("Re(s)")
eixos[0].set_ylabel("Im(s)")

eixos[1].set_xlim(0, 8)
eixos[1].set_title("Resposta ao degrau (ganho DC = 1)")
eixos[1].set_xlabel("t (s)")
eixos[1].set_ylabel("y(t)")

for eixo in eixos:
    eixo.legend()
    eixo.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 13. ANÁLISE NUMÉRICA: POLOS, ESTABILIDADE E RESPOSTA
# ============================================================

T_LONGO = 40.0                                 # Horizonte para medir ts (s)
t_longo = np.arange(0, T_LONGO, dt)

sistemas = [
    ("p1 = -1", [P1], None),
    ("p2 = -5", [P2], None),
    ("p3 = +1", [P3], 1.0),
]

for sigma in lista_sigma:
    sistemas.append((f"par sigma={sigma:g}, w={OMEGA_FIXO:g}",
                     par_complexo(sigma, OMEGA_FIXO), None))

for omega in lista_omega:
    if omega != OMEGA_FIXO or SIGMA_FIXO != 1.0:
        sistemas.append((f"par sigma={SIGMA_FIXO:g}, w={omega:g}",
                         par_complexo(SIGMA_FIXO, omega), None))

sistemas.append(("composto (sem p3)", polos_estavel, None))
sistemas.append(("composto (com p3)", polos_instavel, None))

print("ANÁLISE DE POLOS, ZEROS E ESTABILIDADE")
print("=" * 118)
print(f"{'Sistema':<24}{'Polos':<38}{'Estabilidade':<15}{'y(12 s)':>12}"
      f"{'ts 2% (s)':>11}{'4/|Re|':>9}{'Mp (%)':>9}{'Mp teor.':>10}")

for nome, polos, K in sistemas:
    y = degrau(polos, K=K)
    classe = classificar_estabilidade(polos)

    if classe == "ESTÁVEL":
        num_s, den_s = construir_sistema(polos, K=K)
        y_longo = resposta_degrau(t_longo, num_s, den_s)

        ts = tempo_acomodacao(t_longo, y_longo)
        re_dominante = min(abs(np.real(p)) for p in polos)
        mp = max(0.0, (y_longo.max() - 1.0) * 100)

        txt_ts = f"{ts:11.3f}"
        txt_4 = f"{4 / re_dominante:9.3f}"
        txt_mp = f"{mp:9.2f}"
    else:
        txt_ts = f"{'-':>11}"
        txt_4 = f"{'-':>9}"
        txt_mp = f"{'-':>9}"

    # Sobressinal teórico apenas para um par complexo isolado
    if len(polos) == 2 and abs(np.imag(polos[0])) > 1e-9 and classe == "ESTÁVEL":
        sigma = -np.real(polos[0])
        omega = abs(np.imag(polos[0]))
        txt_teor = f"{100 * np.exp(-np.pi * sigma / omega):10.2f}"
    else:
        txt_teor = f"{'-':>10}"

    print(f"{nome:<24}{texto_polos(polos):<38}{classe:<15}{y[-1]:12.4g}"
          f"{txt_ts}{txt_4}{txt_mp}{txt_teor}")


print("\n\nEfeito dos zeros (polos em -1 e -5, ganho DC = 1)")
print("=" * 70)
print(f"{'Zero':<14}{'Mp (%)':>10}{'Subsolavanco':>15}{'ts 2% (s)':>12}")

for zero in lista_zeros:
    zeros_caso = [] if zero is None else [zero]
    nome = "sem zero" if zero is None else f"{zero:+g}"

    num_z, den_z = construir_sistema([P1, P2], zeros_caso)
    y = resposta_degrau(t_longo, num_z, den_z)

    mp = max(0.0, (y.max() - 1.0) * 100)
    subsolavanco = max(0.0, -y.min())
    ts = tempo_acomodacao(t_longo, y)

    print(f"{nome:<14}{mp:10.2f}{subsolavanco:15.4f}{ts:12.3f}")


# ============================================================
# 14. VERIFICAÇÃO: SOLUÇÃO ANALÍTICA x SIMULAÇÃO RK4
# ============================================================

print("\n\nVerificação: analítico x Runge-Kutta (resposta ao degrau)")
print("=" * 70)
print(f"{'Sistema':<28}{'Erro máximo':>14}{'Erro relativo':>16}")

casos = [(nome, polos, [], K) for nome, polos, K in sistemas]

for zero in lista_zeros:
    if zero is not None:
        casos.append((f"polos -1,-5 e zero {zero:+g}", [P1, P2], [zero], None))

for nome, polos, zeros, K in casos:
    num, den = construir_sistema(polos, zeros, K)

    y_analitico = resposta_degrau(t, num, den)
    y_numerico = simular_rk4(num, den, t)

    erro = np.max(np.abs(y_analitico - y_numerico))
    erro_relativo = erro / np.max(np.abs(y_analitico))

    print(f"{nome:<28}{erro:14.3e}{erro_relativo:16.3e}")


# ============================================================
# 15. RELAÇÃO ENTRE POSIÇÃO DOS POLOS E COMPORTAMENTO TEMPORAL
# ============================================================

print("\n\nCONCLUSÕES")
print("=" * 70)
print("Cada polo p_k contribui com um termo exp(p_k * t) na resposta:")
print("  Re{p_k} < 0 : termo decrescente; quanto mais à esquerda, mais rápido")
print("                (constante de tempo 1/|Re{p_k}|, ts ~ 4/|Re{p_k}|)")
print("  Re{p_k} > 0 : termo crescente; basta um polo para o sistema ser instável")
print("  Re{p_k} = 0 : termo sem decaimento (oscilação sustentada se Im != 0)")
print("  Im{p_k} != 0: oscilação de frequência |Im{p_k}| rad/s, sempre em pares")
print("                conjugados (período T = 2*pi/|Im{p_k}|)")
print("  Polo mais próximo do eixo j*omega (dominante) domina o regime lento.")
print("  Zeros não alteram a estabilidade, mas modificam a forma da resposta")
print("  (sobressinal ou subsolavanco, este último com zeros em Re > 0).")

print("\nSimulação concluída!")